# HW04 Data Science Academy
# Мурзабеков Эльдар 

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.

In [1]:
import numpy as np 

1. Создаем массив и делим на 2 и на 4 без остатка, результат 1, 2, 3 затем 0, 1, 1

In [2]:
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)

[1. 2. 3.] [0 1 1]


2. Матрица из единиц с 3 строками и 4 столбцами, через axis=1 суммируем только столбцы. Прогноз 4 4 4

In [3]:
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))

[4. 4. 4.]


3. Складываем горизонтальный вектор и вертикальный поэлементно, прогноз: [0, 1, 2] [1, 2, 3] [2, 3, 4] 

In [4]:
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))

[[0 1 2]
 [1 2 3]
 [2 3 4]]


4. Суммируем квадратный корень массива, прогноз: 6

In [5]:
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())

6.0


5. Берем среднее с массива квадратов от 1 по 4, прогноз: 7.5

In [6]:
# 5
print((np.arange(1, 5) ** 2).mean())

7.5


In [7]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [8]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [9]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [10]:
print(f"Чистые оплаченные заказы: {clean.sum()}")
sep, oct = np.sum(amount[clean & (month == '2026-09')]), np.sum(amount[clean & (month == '2026-10')])
print(f"Выручка за сентябрь: {sep}")
print(f"Выручка за октябрь: {oct}")
print(f"Изменение в {(oct - sep) / sep * 100:.2f}%")


Чистые оплаченные заказы: 4739
Выручка за сентябрь: 35243410.0
Выручка за октябрь: 34419520.0
Изменение в -2.34%


После удаления повторных заказов и тестового клиента выручка упала на 2,34%. В ДЗ 2 после очистки получилось примерно столько же 2,3%

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [11]:
categories = np.unique(cat)
months = np.array(['2026-09', '2026-10'])
table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months] for c in categories])
for c, m in zip(categories, table):
    print(f"{c:<10} {m[0]:>12,.0f} {m[1]:>12,.0f}")
print(table.shape)
print(f'Сумма категорий: {table.sum(axis=1)}')
print(f'Изменение октября относительно сентября: {table[:, 1] / table[:, 0] - 1}')

Дом           4,997,730    5,125,790
Книги         1,065,300    1,140,910
Красота       2,610,950    2,678,990
Одежда        9,167,100    8,505,730
Электроника   17,402,330   16,968,100
(5, 2)
Сумма категорий: [10123520.  2206210.  5289940. 17672830. 34370430.]
Изменение октября относительно сентября: [ 0.02562363  0.07097531  0.02605948 -0.07214604 -0.02495241]


1. Больше всего за два месяца удалось заработать на продаже электроники 34 370 430 тг
2. В категории одежды выручка снизилась сильнее всего на 7,21%

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [34]:
orders = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months] for c in categories])
print(orders)
avg = table / orders
print(avg)
print(f"Самая высокая категория по среднему чеку: {categories[avg[:, 1].argmax()]}")

[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
[[10499.43277311 10768.46638655]
 [ 4020.          3815.75250836]
 [ 6187.08530806  6158.59770115]
 [13520.79646018 13045.59815951]
 [32772.74952919 33600.1980198 ]]
Самая высокая категория по среднему чеку: Электроника


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [56]:
cities = np.unique(city)
city_table = np.array([[amount[(clean & (city == c) & (month == m))].sum() for m in months] for c in cities])
print(city_table)
city_share = city_table / city_table.sum(axis=0)
print(f"Каждый столбец в сумме дает: {city_share.sum(axis=0)}")
print(f"Доля Шымкента: {city_share[cities == 'Шымкент']}")
print(f"Изменение доли Шымкента: {(city_share[cities == 'Шымкент'][0, 1] - city_share[cities == 'Шымкент'][0, 0]) * 100:.2f} п.п.")

[[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]
Каждый столбец в сумме дает: [1. 1.]
Доля Шымкента: [[0.12934759 0.11749176]]
Изменение доли Шымкента: -1.19 п.п.


Доля Шымкента уменьшилась на 1,19 процентного пункта по сравнению с первым месяцем

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [69]:
city_change = city_table[:, 1] - city_table[:, 0]
total_change = np.sum(city_table[:, 1] - city_table[:, 0])
print(f"Изменение выручки каждого города: {city_change}")
print(f"Общее изменение выручки: {total_change}")
print(f"Процент падения Шымкента: {city_change[cities == 'Шымкент'][0] / total_change * 100:.2f}%")

Изменение выручки каждого города: [ -36540.  141950. -183690.   45870. -276840. -514640.]
Общее изменение выручки: -823890.0
Процент падения Шымкента: 62.46%


Основное снижение приходится на Шымкент, который формирует почти 62,5% от общего падения. Также стоит обратить внимание на Караганду и Астану.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [72]:
share = table / table.sum(axis=1, keepdims=True)
print(share)

[[0.49367512 0.50632488]
 [0.48286428 0.51713572]
 [0.49356893 0.50643107]
 [0.51871149 0.48128851]
 [0.50631691 0.49368309]]


В первом варианте размеры массивов не совпадали, поэтому возникала ошибка. С keepdims=True размерность сохраняется, и NumPy может растянуть значения для деления каждой строки на её сумму.

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [98]:
x = amount[clean]
z = (x - x.mean()) / x.std()
print(f"Z оценка: {z}")
m3 = np.abs(z) > 3
print(f"Количество больше 3: {m3.sum()}")
print(f"Доля: {m3.sum() / len(z) * 100:.2f}")
print(f"Среднее z: {z.mean()}")
print(f"Стандартное отклонение: {z.std()}")

Z оценка: [-0.49880681 -0.83285331 -0.41980933 ... -0.83210095  0.56051183
  0.01580537]
Количество больше 3: 87
Доля: 1.84
Среднее z: 7.04695264417065e-17
Стандартное отклонение: 1.0


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [109]:
valid = clean & ~np.isnan(days)
print(valid.sum())
X = np.column_stack((amount[valid], days[valid]))
print(X)
X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print(X_scaled)
print(X_scaled.min(axis=0))
print(X_scaled.max(axis=0))

4595
[[8.070e+03 1.000e+00]
 [3.630e+03 2.000e+00]
 [9.120e+03 3.000e+00]
 ...
 [1.143e+04 2.000e+00]
 [3.640e+03 2.000e+00]
 [1.491e+04 2.000e+00]]
[[0.05313765 0.        ]
 [0.02103817 0.14285714]
 [0.06072874 0.28571429]
 ...
 [0.07742915 0.14285714]
 [0.02111047 0.14285714]
 [0.1025882  0.14285714]]
[0. 0.]
[1. 1.]


Масштабирование приводит все признаки к диапазону от 0 до 1. Это нужно, чтобы большие числа не влияли на обучение модели сильнее, чем маленькие, только из-за своей величины.